# 显存去哪了：tensor 与 nvidia-smi

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-04-where-did-vram-go.ipynb)

动手观察显存。边在 GPU 上分配 tensor、边盯 nvidia-smi，看清 PyTorch 的 caching allocator 与 nvidia-smi 的关系，这是理解 GPU 共享时显存配额与隔离的物理基础。

本 notebook 配合教程章节 [显存去哪了：tensor 与 nvidia-smi](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

[Lab 01](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-01-meet-your-gpu/) 里我们看到 `Memory-Usage: 0MiB / 15360MiB`。当你开始分配 tensor，显存会涨，但涨的方式可能出乎意料。理解 **“PyTorch 已经认为释放了、但 `nvidia-smi` 还显示占用”** 这一现象，是理解共享 GPU 时为什么需要显存配额、以及配额实际在管什么的物理基础。

## 第 1 步：基线

In [ ]:
import torch, subprocess

def gpu_mem_mib():
    out = subprocess.check_output(["nvidia-smi","--query-gpu=memory.used","--format=csv,noheader,nounits"]).decode().strip()
    return int(out.splitlines()[0])

print("before any CUDA call :", gpu_mem_mib(), "MiB")
torch.cuda.init(); torch.zeros(1, device="cuda")   # 强制创建 CUDA context
print("after context init   :", gpu_mem_mib(), "MiB")

实测输出：

```text
before any CUDA call : 3 MiB
after context init   : 123 MiB
```

第一行接近 0：`import torch` 不会碰 GPU，剩下的 3 MiB 是驱动自己的占用。第二行涨了约 120 MiB，这就是 **CUDA context**：驱动为这个进程在显存里建的页表、kernel 镜像（kernel 指交给 GPU 执行的函数，不是 Linux 内核，[Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 会解释）等运行环境。它不属于任何 tensor，也不会被 `empty_cache()` 释放，**每个用 GPU 的进程都要交这笔“入场费”**。

120 MiB 是最小值：新版 CUDA 默认按需加载 kernel，进程第一次用到 cuBLAS（矩阵乘）、cuDNN（卷积）时还会再涨几十到几百 MiB。

> **context 开销为什么对共享很重要**
>
> 如果 10 个 Pod 共享一张 T4，光 context 就要占用 1.2 GB 以上，用到 cuBLAS / cuDNN 后会更多。[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 会实测：即使开了 MPS，每个进程的这份显存也省不掉，所以共享 GPU 时要把它算进配额。

## 第 2 步：分配一个大 tensor

In [ ]:
# 2 GiB 的 float32（2*1024*1024*1024 字节）
a = torch.randn(2 * 1024 * 1024 * 1024 // 4, device="cuda")
print("after alloc  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees allocated   :", round(torch.cuda.memory_allocated()/1024**2), "MiB")
print("torch sees reserved    :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

实测输出：

```text
after alloc  nvidia-smi: 2171 MiB
torch sees allocated   : 2048 MiB
torch sees reserved    : 2050 MiB
```

三个数字是三种统计方式：

- `nvidia-smi` 2171 MiB：驱动视角的总占用，等于 context 123 MiB 加上 2048 MiB 的 tensor；
- `allocated` 2048 MiB：正在被 tensor 使用的显存；
- `reserved` 2050 MiB：PyTorch 向驱动要来、攥在手里的显存。多出的 2 MiB 是第 1 步那个 `torch.zeros(1)` 占的最小块。PyTorch 按 2 MiB / 20 MiB 的块向驱动要显存，分配许多小 tensor 时 `reserved` 会明显大于 `allocated`。

三者是一层套一层的关系，外层的读数永远不小于内层：

![三种显存统计方式：nvidia-smi 看到整个进程，包含 CUDA context 和 PyTorch 的 reserved；reserved 里正在被 tensor 使用的部分才是 allocated](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/vram-three-views.svg)

## 第 3 步：删掉 tensor，显存为什么不立刻还？

In [ ]:
del a
print("after del  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees allocated :", round(torch.cuda.memory_allocated()/1024**2), "MiB")
print("torch sees reserved  :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

实测输出：

```text
after del  nvidia-smi: 2171 MiB
torch sees allocated : 0 MiB
torch sees reserved  : 2050 MiB
```

对比第 2 步：

- `allocated` 立刻归零，PyTorch 认为 tensor 已释放；
- 但 `nvidia-smi` 和 `reserved` 一点没掉：PyTorch 的 **caching allocator** 把这块显存攥在手里，准备下次复用，而不是立刻还给驱动。

这几块显存分别归谁用、什么时候占、什么时候才能放，看下面这张框图。实线是占用，虚线是释放：

![显存归属框图：驱动管理空闲显存；进程首次调用 CUDA 时创建 context，进程退出才归还；PyTorch 缓存不够时向驱动 cudaMalloc，创建 tensor 时从缓存切出已分配块，del 后退回缓存成为空闲块，只有 empty_cache 才把空闲块还给驱动](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/vram-ownership.svg)

## 第 4 步：强制还给驱动

In [ ]:
torch.cuda.empty_cache()
print("after empty_cache  nvidia-smi:", gpu_mem_mib(), "MiB")
print("torch sees reserved        :", round(torch.cuda.memory_reserved()/1024**2), "MiB")

实测输出：

```text
after empty_cache  nvidia-smi: 121 MiB
torch sees reserved        : 0 MiB
```

`nvidia-smi` 回落到第 1 步的 context 基线（约 120 MiB），`reserved` 归零。`empty_cache()` 只释放 caching allocator 的空闲块，**不影响**正在使用的 tensor，也**不会**释放 context。

把四步串起来看，三个读数的变化就一目了然：

![显存的一生：基线只有 context；分配后 allocated 上涨；del 后 tensor 显存留在缓存里，nvidia-smi 不变；empty_cache 后才还给驱动](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/vram-lifecycle.svg)

> **为什么这件事很重要**
>
> 在真实多租户平台里，多个 Pod 共用一张 GPU。如果某个进程的 caching allocator 攥着 2 GB 不还，别的进程（Pod）就拿不到这 2 GB。**共享 GPU 的显存隔离管的就是这一层：按驱动统计的“真实占用”给每个 Pod 设上限**，而不是相信每个进程都会自觉 `empty_cache`。实现方式有几类：MIG 在硬件上切出独立显存；HAMi-core 在 CUDA API 层拦截分配；CUDA 13.4 起驱动也提供了显存上限能力，KAI Scheduler 的 GPU 分数方案已在使用。这是你在 Colab 单机就能亲眼看到的物理问题。

## 验证

完成后你应该能解释：

- 为什么 `del a` 之后 `nvidia-smi` 的显存可能不掉；
- `memory_allocated` 与 `memory_reserved` 的区别；
- `empty_cache()` 做了什么、不做什么；
- 这和共享 GPU 时的显存隔离有什么关系。

## 故障模式



| 症状 | 原因 | 处理 |
| :--- | :--- | :--- |
| 分配时报 `CUDA out of memory` | 之前 session 残留 / 要得太接近 15 GB | 重启 runtime，或减小 tensor |
| `del` 后 `nvidia-smi` 没掉 | caching allocator 正常行为 | `torch.cuda.empty_cache()` |
| `empty_cache` 后显存仍有约 120 MiB | CUDA context 常驻 | 正常，进程退出才会归零 |
| `del a` 后 `memory_allocated` 没降 | notebook 里还有变量引用它（如 `_` 或输出历史） | 避免在 cell 末尾裸写 `a`；必要时 `import gc; gc.collect()` |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。